In [ ]:
# Colab setup: install pyedautils; elsewhere load plotly.js from a CDN for the plots
import sys
if "google.colab" in sys.modules:
    %pip install -q "pyedautils>=0.0.32"
else:
    import plotly.io as pio
    pio.renderers.default = "notebook_connected"

# Data Quality

Before performing any analysis, it is essential to assess and improve the quality of the data. Real-world sensor data often contains missing values, gaps and duplicates.

This chapter covers common techniques for detecting and handling these issues. Implausible values (outliers) are covered in the recipe [Outlier Detection](../partDataVis/statisticalOutliers.ipynb).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatTempHum.csv"

df = pd.read_csv(url, sep=";")
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time")
df.head()

## Missing Values

Missing values (`NaN`) are common in sensor data due to communication failures, sensor outages, or maintenance periods.

In [ ]:
# Count missing values per column
df.isna().sum()

In [ ]:
# Percentage of missing values
missing_pct = (df.isna().sum() / len(df)) * 100
missing_pct.round(2)

In [ ]:
# Detailed info including non-null counts
df.info()

### Visualizing Missing Data

A heatmap of missing values helps identify patterns — for example, whether all sensors fail at the same time.

In [ ]:
# Resample to daily resolution for a cleaner visualization
missing_daily = df.isna().resample("D").mean() * 100

fig = px.imshow(
    missing_daily.T,
    labels=dict(x="Date", y="Column", color="% Missing"),
    title="Missing Data Pattern (% per day)",
    color_continuous_scale="Reds",
    aspect="auto"
)
fig.show()

## Detecting Gaps in Time Series

A gap shows up in one of two ways: as **missing timestamps** (no row at all), or as rows whose values are `NaN`. Check both.

In [ ]:
# 1) Missing timestamps: expected vs. actual number of hourly rows
expected = pd.date_range(df.index.min(), df.index.max(), freq="h")
print(f"Expected timestamps: {len(expected)}")
print(f"Actual timestamps:   {len(df)}")

In [ ]:
# 2) Gaps as NaN rows: length of each run of missing values
s = df["FlatA_Temp"]
gap_len = s.isna().groupby(s.notna().cumsum()).sum()  # NaN rows after each value
gap_len = gap_len[gap_len > 0]
print(f"FlatA_Temp: {len(gap_len)} gaps, longest {gap_len.max()} hours, "
      f"{(gap_len > 6).sum()} longer than 6 hours")

## Duplicates

Duplicate rows can occur when data is merged from multiple sources or exported incorrectly.

In [ ]:
# df.duplicated() compares the values only, not the timestamp
dup_rows = df.duplicated()
empty_rows = df.isna().all(axis=1)
print(f"Rows equal to an earlier row: {dup_rows.sum()}")
print(f"... of which completely empty (all sensors NaN): {(dup_rows & empty_rows).sum()}")

In [ ]:
# Duplicate timestamps are what matters for time series
print(f"Number of duplicate timestamps: {df.index.duplicated().sum()}")

In [ ]:
# Remove duplicates (if any)
df_clean = df[~df.index.duplicated(keep="first")]
print(f"Rows before: {len(df)}, after: {len(df_clean)}")

```{tip}
Use `keep="first"` to keep the first occurrence, `keep="last"` to keep the last, or `keep=False` to remove all duplicates.
```

## Interpolation and Filling Missing Values

Once missing values are identified, they can be filled using various strategies.

### Linear Interpolation

In [ ]:
col = "FlatA_Temp"

# Original data with NaN values
print(f"NaN values before: {df[col].isna().sum()}")

# Interpolate linearly, only between two valid values (limit_area="inside")
df_interp = df.copy()
df_interp[col] = df_interp[col].interpolate(method="linear", limit_area="inside")

# The NaN values left are at the end of the series, where nothing can be interpolated
print(f"NaN values after:  {df_interp[col].isna().sum()}")

Interpolation also bridges long gaps with a straight line. To fill only short gaps, see
*Gap filling strategies* in [Time Series](timeSeries.ipynb).

In [ ]:
# Visualize a section with interpolated values
# Find a region with NaN values
nan_mask = df[col].isna()
if nan_mask.any():
    first_nan = df[nan_mask].index[0]
    start = first_nan - pd.Timedelta(days=2)
    end = first_nan + pd.Timedelta(days=2)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df.loc[start:end].index, y=df.loc[start:end, col],
        mode="lines+markers", name="Original", line=dict(color="steelblue")
    ))
    fig.add_trace(go.Scatter(
        x=df_interp.loc[start:end].index, y=df_interp.loc[start:end, col],
        mode="lines", name="Interpolated", line=dict(color="orange", dash="dash")
    ))
    fig.update_layout(title="Linear Interpolation of Missing Values", xaxis_title="Time", yaxis_title="Temperature [°C]")
    fig.show()

### Forward and Backward Fill

In [ ]:
# Forward fill: carry the last valid observation forward
df_ffill = df.copy()
df_ffill[col] = df_ffill[col].ffill()

# Backward fill: carry the next valid observation backward
df_bfill = df.copy()
df_bfill[col] = df_bfill[col].bfill()

print(f"Forward fill NaN remaining: {df_ffill[col].isna().sum()}")
print(f"Backward fill NaN remaining: {df_bfill[col].isna().sum()}")

### Fill with a Constant

In [ ]:
# Fill with the column mean
df_mean_fill = df.copy()
df_mean_fill[col] = df_mean_fill[col].fillna(df[col].mean())

print(f"NaN remaining: {df_mean_fill[col].isna().sum()}")

```{note}
Choose the imputation method carefully based on the nature of the data. Linear interpolation works well for slowly changing sensor data. Forward fill is appropriate when the last known value is the best estimate. Filling with the mean is generally not recommended for time series data.
```

## Summary Statistics for Quality Assessment

A quick quality overview can be compiled by combining several checks.

In [ ]:
quality_summary = pd.DataFrame({
    "total_rows": len(df),
    "non_null": df.count(),
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().sum() / len(df) * 100).round(2),
    "min": df.min(),
    "max": df.max(),
    "mean": df.mean().round(2),
    "std": df.std().round(2)
})

quality_summary

In [ ]:
# Visualize data completeness
completeness = (1 - df.isna().mean()) * 100

fig = px.bar(
    x=completeness.index,
    y=completeness.values,
    labels={"x": "Column", "y": "Completeness (%)"},
    title="Data Completeness per Column"
)
fig.update_layout(yaxis_range=[0, 105])
fig.show()